In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf
from IPython.display import display, HTML

clean = Path("data/clean")
results = Path("results")
results.mkdir(exist_ok=True)

# -----------------------------
# Load combined price + population data
# -----------------------------

wide = pd.read_csv(clean / "population_density_all_suburbs.csv")

wide["suburb"] = wide["suburb"].astype(str).str.strip().str.upper()
wide["treated"] = wide["treated"].astype(int)

# -----------------------------
# Reshape price and population columns to long format
# -----------------------------

price_cols = [col for col in wide.columns if col.endswith("_price")]
pop_cols = [col for col in wide.columns if col.endswith("_pop")]

prices_long = wide[["suburb", "status", "treated"] + price_cols].melt(
    id_vars=["suburb", "status", "treated"],
    value_vars=price_cols,
    var_name="year",
    value_name="price"
)

prices_long["year"] = prices_long["year"].str.replace("_price", "", regex=False).astype(int)

population_long = wide[["suburb"] + pop_cols].melt(
    id_vars=["suburb"],
    value_vars=pop_cols,
    var_name="year",
    value_name="population"
)

population_long["year"] = population_long["year"].str.replace("_pop", "", regex=False).astype(int)

# -----------------------------
# Merge final panel
# -----------------------------

df = (
    prices_long
    .merge(population_long, on=["suburb", "year"], how="inner")
    .dropna(subset=["price", "population", "treated"])
    .copy()
)

df = df[(df["price"] > 0) & (df["population"] > 0)].copy()

# -----------------------------
# DiD variables
# -----------------------------

df["post"] = (df["year"] >= 2017).astype(int)
df["treat_post"] = df["treated"] * df["post"]

df["log_price"] = np.log(df["price"])
df["log_population"] = np.log(df["population"])

# -----------------------------
# Run DiD regression
# -----------------------------

model = smf.ols(
    "log_price ~ treat_post + log_population + C(suburb) + C(year)",
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["suburb"]}
)

# -----------------------------
# Build table
# -----------------------------

def get_row(label, variable):
    if variable not in model.params.index:
        return [label, "Absorbed", "", "", "", ""]

    coef = model.params[variable]
    se = model.bse[variable]
    t_val = model.tvalues[variable]
    p_val = model.pvalues[variable]
    ci_low, ci_high = model.conf_int().loc[variable]

    return [
        label,
        f"{coef:.3f}",
        f"{se:.3f}",
        f"{t_val:.3f}",
        f"{p_val:.3f}",
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    ]

did_table = pd.DataFrame(
    [
        get_row("Intercept", "Intercept"),
        ["Treat", "Absorbed by suburb FE", "", "", "", ""],
        ["Post", "Absorbed by year FE", "", "", "", ""],
        get_row("Treat x Post (DiD effect)", "treat_post"),
        get_row("log_population", "log_population"),
        ["Suburb FE", "Yes", "", "", "", ""],
        ["Year FE", "Yes", "", "", "", ""],
        ["N", f"{int(model.nobs):,}", "", "", "", ""],
        ["R²", f"{model.rsquared:.3f}", "", "", "", ""],
    ],
    columns=["Variable", "Coef", "Std Err", "t", "p-value", "CI"]
)

# -----------------------------
# Display formatted table
# -----------------------------

html = did_table.to_html(index=False, escape=False)

html = html.replace(
    '<table border="1" class="dataframe">',
    '<table style="border-collapse:separate; border-spacing:0; width:92%; font-family:Arial, sans-serif; font-size:16px; color:#2b2927; border:1px solid #dcc7b3; border-radius:10px; overflow:hidden;">'
)

html = html.replace(
    "<thead>",
    '<thead style="background-color:#f7eee5;">'
)

html = html.replace(
    "<th>",
    '<th style="padding:14px 18px; text-align:left; font-weight:700; border-right:1px solid #dcc7b3; border-bottom:1px solid #dcc7b3;">'
)

html = html.replace(
    "<td>",
    '<td style="padding:14px 18px; text-align:left; border-right:1px solid #dcc7b3; border-bottom:1px solid #dcc7b3;">'
)

display(HTML(html))

# -----------------------------
# Save table
# -----------------------------

did_table.to_csv(results / "new_did_regression_table.csv", index=False)

with open(results / "new_did_regression_table.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Saved:")
print(results / "new_did_regression_table.csv")
print(results / "new_did_regression_table.html")

# -----------------------------
# Interpretation values
# -----------------------------

did_coef = model.params["treat_post"]
did_se = model.bse["treat_post"]
did_p = model.pvalues["treat_post"]
did_pct = (np.exp(did_coef) - 1) * 100

print()
print("Sample used:")
print("Suburbs:", df["suburb"].nunique())
print("Observations:", len(df))
print("Treated suburbs:", df.loc[df["treated"] == 1, "suburb"].nunique())
print("Control suburbs:", df.loc[df["treated"] == 0, "suburb"].nunique())

print()
print("Main DiD result:")
print(f"Treat x Post coefficient: {did_coef:.3f}")
print(f"Clustered standard error: {did_se:.3f}")
print(f"p-value: {did_p:.3f}")
print(f"Approximate percentage effect: {did_pct:.1f}%")


Variable,Coef,Std Err,t,p-value,CI
Intercept,10.755,0.803,13.391,0.000,"[9.181, 12.329]"
Treat,Absorbed by suburb FE,,,,
Post,Absorbed by year FE,,,,
Treat x Post (DiD effect),-0.015,0.030,-0.510,0.610,"[-0.075, 0.044]"
log_population,0.290,0.085,3.403,0.001,"[0.123, 0.457]"
Suburb FE,Yes,,,,
Year FE,Yes,,,,
N,490,,,,
R²,0.982,,,,


Saved:
results/new_did_regression_table.csv
results/new_did_regression_table.html

Sample used:
Suburbs: 49
Observations: 490
Treated suburbs: 24
Control suburbs: 25

Main DiD result:
Treat x Post coefficient: -0.015
Clustered standard error: 0.030
p-value: 0.610
Approximate percentage effect: -1.5%


## Analysis Declaration

This is a **causal analysis** aiming to identify the treatment effect of LXRP on house prices. We declare this as causal and will be evaluated against causal standards, including identification strategy, assumptions, and threats to validity.

## Econometric Specification

### Functional Form
We estimate the following regression model with fixed effects:

\[
\log(\text{price}_{it}) = \alpha_i + \gamma_t + \delta (\text{Treat}_i \times \text{Post}_t) + \beta \log(\text{pop}_{it}) + \varepsilon_{it}
\]

Where:
- $\log(\text{price}_{it})$: Natural logarithm of median house prices in suburb $i$ at time $t$
- $\alpha_i$: Suburb fixed effects (absorb time-invariant differences between suburbs, including the main Treat effect)
- $\gamma_t$: Year fixed effects (absorb common time shocks, including the main Post effect)
- $\text{Treat}_i \times \text{Post}_t$: DiD interaction term capturing the treatment effect
- $\log(\text{pop}_{it})$: Natural logarithm of population in suburb $i$ at time $t$
- $\varepsilon_{it}$: Idiosyncratic error term
### Regressors
- Main regressors: Treat, Post, DiD interaction, log(population)
- Additional controls: Suburb fixed effects and year fixed effects (absorbed in the reported specification)

### Sample
- Melbourne suburbs with available house price and population data from 2015-2024
- Treated group: Suburbs within 2 km of LXRP sites
- Control group: Other suburbs in the sample
- Total observations: 490 (after cleaning)

### Error Structure
- Standard errors clustered at the suburb level to account for serial correlation in panel data
- Justification: Clustering addresses potential correlation within suburbs over time, providing robust inference

## Identification Strategy (Causal)

We employ **Difference-in-Differences (DiD)** as our identification strategy. This method compares the change in outcomes over time between treated and control groups.

## Identification Strategy (Causal)

We employ **Difference-in-Differences (DiD)** as our identification strategy. This method compares the change in outcomes over time between treated and control groups.

## Interpretation of Main Coefficients

- **DiD Effect (-0.015)**: The coefficient indicates that, holding population constant, house prices in treated suburbs were about 1.5% lower after 2017 relative to control suburbs. This estimate is not statistically significant.
- **Magnitude and Units**: The effect is in log points, so the implied change is small and economically negligible in this specification.
- **Statistical Inference**: With a p-value of 0.610, the data do not provide evidence of a reliable treatment effect after 2017.
- **Conclusion**: The regression is consistent with no meaningful change in house prices for treated suburbs after 2017 once fixed effects and population controls are included.

## Threats to Validity

As a causal analysis, we address potential threats:

- **Omitted Variable Bias (OVB)**: Other urban developments or policy changes in treated suburbs could confound the results. *Mitigation*: We include population controls and fixed effects to account for observable differences; parallel trends assumption helps control for unobservables.
- **Selection Bias**: Treated suburbs might differ systematically from controls (e.g., pre-existing price trends). *Mitigation*: Distance-based treatment assignment reduces selection concerns; pre-treatment parallel trends tested (though not shown here).
- **Reverse Causality**: Unlikely, as LXRP sites are predetermined infrastructure projects, not driven by house prices.
- **Measurement Error**: House price data may have reporting biases. *Mitigation*: Use of median prices and consistent data sources.